# 09 Production Hardening and Release Gates (OpenClaw, 2026)

## What This Lesson Is
Define objective release gates for OpenClaw deployments across safety, reliability, and cost.

## Scientific Lens
- Concept: Go/no-go decisions must be evidence-based and reproducible.
- Measure: Weighted release score and pass/fail decision consistency.
- Validity Limit: Passing gates reduces but does not eliminate production risk.


## How It Works
1. Define weighted gate model spanning policy, auth, telemetry, and rollback.
2. Compute deterministic release decision.
3. Run live OpenClaw call to generate operational hardening checklist for comparison.


In [ ]:
import os
from openai import OpenAI  # OpenAI SDK used as protocol client to OpenClaw gateway

OPENCLAW_BASE_URL = os.getenv("OPENCLAW_BASE_URL", "http://127.0.0.1:18789").rstrip("/")
OPENCLAW_GATEWAY_TOKEN = os.getenv("OPENCLAW_GATEWAY_TOKEN") or os.getenv("OPENAI_API_KEY") or ""
OPENCLAW_TOKEN_SOURCE = (
    "OPENCLAW_GATEWAY_TOKEN" if os.getenv("OPENCLAW_GATEWAY_TOKEN")
    else ("OPENAI_API_KEY" if os.getenv("OPENAI_API_KEY") else "<missing>")
)
OPENCLAW_AGENT_ID = os.getenv("OPENCLAW_AGENT_ID", "main")

print("OPENCLAW_BASE_URL:", OPENCLAW_BASE_URL)
print("OPENCLAW_GATEWAY_TOKEN source:", OPENCLAW_TOKEN_SOURCE)
print("OPENCLAW_AGENT_ID:", OPENCLAW_AGENT_ID)


def build_gateway_client() -> OpenAI:
    # OpenClaw exposes an OpenAI-compatible Chat Completions endpoint at /v1/chat/completions.
    # We use the OpenAI SDK as a transport/protocol client to OpenClaw (not directly to OpenAI).
    # OpenClaw then routes to configured downstream providers/models.
    # Docs: https://docs.openclaw.ai/gateway/openai-http-api
    return OpenAI(base_url=f"{OPENCLAW_BASE_URL}/v1", api_key=OPENCLAW_GATEWAY_TOKEN or "local-dev-token")


def ask_openclaw(prompt: str, user: str = "lesson-user", temperature: float = 0.2) -> str:
    client = build_gateway_client()
    resp = client.chat.completions.create(
        model="openclaw",  # gateway-level alias/router target
        messages=[{"role": "user", "content": prompt}],
        user=user,
        temperature=temperature,
        extra_headers={"x-openclaw-agent-id": OPENCLAW_AGENT_ID},
    )
    return resp.choices[0].message.content or ""


### Why This Uses `OpenAI` Client With `model="openclaw"`
- The `OpenAI` SDK here is used as a **protocol-compatible HTTP client**.
- Requests go to the **OpenClaw gateway** (`OPENCLAW_BASE_URL/v1`), not directly to OpenAI.
- `model="openclaw"` is a **gateway alias/router target**.
- OpenClaw establishes downstream provider connections (OpenAI/Ollama/etc.) based on its own model/policy config.


## Code Walkthrough
- `Deterministic Demo` defines and validates the decision logic.
- `Live Demo` executes a real OpenClaw agent call through the OpenAI-compatible gateway API.


In [ ]:
# Deterministic Demo
gates = {
    "auth_enforced": (1.0, 3),
    "fallback_configured": (1.0, 2),
    "telemetry_enabled": (0.5, 2),
    "rollback_runbook": (1.0, 2),
    "cost_guardrails": (0.5, 1),
}
score = sum(v*w for v,w in gates.values()) / sum(w for _,w in gates.values())
verdict = "approve" if score >= 0.8 else "block"
print(score, verdict)
assert verdict in {"approve", "block"}


In [ ]:
# Live Demo
try:
    q = "Produce a production hardening checklist for OpenClaw with release gates, rollback criteria, and owner mapping."
    print(ask_openclaw(q, user="release-gates"))
except Exception as exc:
    print(f"Live demo call failed: {exc}")
    print("Set OPENCLAW_GATEWAY_TOKEN in .env (or export OPENAI_API_KEY) and rerun.")


## Applied Labs
1. Add minimum threshold per category (security, reliability, governance) instead of single global threshold.
2. Attach evidence pointers to each gate and require non-empty proof.
3. Model rollback trigger matrix by incident severity and response time.

## Validation Checklist
- Release decision is computed from explicit weighted criteria.
- Gate model includes safety + reliability + cost dimensions.
- Live call is used for operational plan augmentation, not as sole decision authority.

## Further Reading
- OpenClaw docs directory: https://docs.openclaw.ai/start/docs-directory
- NIST AI RMF: https://www.nist.gov/itl/ai-risk-management-framework
